<a href="https://colab.research.google.com/github/Swayam-dot-cmd/neural-information-retrieval/blob/main/notebooks/04_hyde_improvement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q beir sentence-transformers transformers scikit-learn

In [2]:
from beir.datasets.data_loader import GenericDataLoader
from beir import util

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

/usr/local/lib/python3.12/dist-packages/beir/datasets/data_loader.py:8: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
dataset = "scifact"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

out_dir = "./datasets"
data_path = util.download_and_unzip(url, out_dir)

corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")

doc_ids = list(corpus.keys())
corpus_texts = [corpus[doc_id]["text"] for doc_id in doc_ids]

print("Corpus:", len(corpus_texts))
print("Queries:", len(queries))

  0%|          | 0/5183 [00:00<?, ?it/s]

Corpus: 5183
Queries: 300


In [4]:
dense_model = SentenceTransformer('BAAI/bge-small-en')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
generator_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

print("Generator loaded")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Generator loaded


In [12]:
def generate_hypo_doc(query):
    prompt = f"""
    Question: {query}

    Write a precise, factual, and scientific paragraph that answers this question.
    Use correct terminology and avoid vague statements.
    """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True)

    outputs = generator_model.generate(
        **inputs,
        max_length=128
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [13]:
corpus_embeddings = dense_model.encode(
    corpus_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Corpus encoded")

Batches:   0%|          | 0/162 [00:00<?, ?it/s]

Corpus encoded


In [18]:
def hyde_retrieve(query, top_k=10):
    hypo_doc = generate_hypo_doc(query)

    query_emb = dense_model.encode([query], convert_to_numpy=True)[0]
    hypo_emb = dense_model.encode([hypo_doc], convert_to_numpy=True)[0]

    # weighted combination
    final_emb = 0.7 * query_emb + 0.3 * hypo_emb

    scores = cosine_similarity([final_emb], corpus_embeddings)[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    return [doc_ids[i] for i in top_indices]

In [19]:
def recall_at_k(retrieved, relevant, k):
    retrieved_k = set(retrieved[:k])
    relevant_set = set(relevant)

    return len(retrieved_k & relevant_set) / len(relevant_set)

In [20]:
recalls = []

for qid in list(queries.keys())[:50]:
    query = queries[qid]

    retrieved_docs = hyde_retrieve(query, top_k=10)
    relevant_docs = list(qrels[qid].keys())

    if len(relevant_docs) == 0:
        continue

    recall = recall_at_k(retrieved_docs, relevant_docs, 10)
    recalls.append(recall)

print("Improved HyDE (50):", sum(recalls)/len(recalls))

Improved HyDE (50): 0.846


In [21]:
recalls = []

for qid in queries:
    query = queries[qid]

    retrieved_docs = hyde_retrieve(query, top_k=10)
    relevant_docs = list(qrels[qid].keys())

    if len(relevant_docs) == 0:
        continue

    recall = recall_at_k(retrieved_docs, relevant_docs, 10)
    recalls.append(recall)

avg_recall = sum(recalls) / len(recalls)

print("Improved HyDE Recall@10:", avg_recall)

Improved HyDE Recall@10: 0.8035555555555556
